In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os
os.environ["OMP_NUM_THREADS"] = "1" # avoid fuite de mémoire sur windows

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [2]:
df = pd.read_csv('c/df.csv')
print('Lines / Columns : ', df.shape)

C:\Users\LN\AppData\Local\Temp\ipykernel_3544\1832951334.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('c/df.csv')


Lines / Columns :  (18713730, 8)


In [3]:
df.head(1)

,pickup_date,lat,lon,dispatching_base_num,locationid,borough,zone,affiliated_base_num
0,2014-04-01 00:11:00,40.769,-73.9549,b02512,140,manhattan,lenox hill east,NaN


In [4]:
df["pickup_date"] = pd.to_datetime(df["pickup_date"])
df['month'] = df["pickup_date"].dt.month
df["hour"] = df["pickup_date"].dt.hour
df["dayofweek"] = df["pickup_date"].dt.dayofweek
df['pickup_date'].dtype


dtype('<M8[ns]')

# Missing values

In [5]:
missing_values = df.isnull().sum()/df.shape[0]*100
print(f"df : \n{missing_values}")

df : 
pickup_date              0.000000
lat                      0.037144
lon                      0.037144
dispatching_base_num     0.000000
locationid               0.000000
borough                  0.000000
zone                     0.000000
affiliated_base_num     23.743268
month                    0.000000
hour                     0.000000
dayofweek                0.000000
dtype: float64


In [6]:
df[df['lon'].isna() | df['lat'].isna()]

,pickup_date,lat,lon,dispatching_base_num,locationid,borough,zone,affiliated_base_num,month,hour,dayofweek
4446897,2015-01-19 10:02:46,NaN,NaN,b02598,265,unknown,unknown,unknown,1,10,0
4453760,2015-03-28 11:39:00,NaN,NaN,b02682,265,unknown,unknown,b02682,3,11,5
4454296,2015-03-28 11:56:00,NaN,NaN,b02682,265,unknown,unknown,b02682,3,11,5
4455088,2015-03-17 08:46:00,NaN,NaN,b02764,57,queens,corona,b02764,3,8,1
4455091,2015-03-17 08:46:00,NaN,NaN,b02764,265,unknown,unknown,b02764,3,8,1
...,...,...,...,...,...,...,...,...,...,...,...
18706255,2015-05-08 10:27:00,NaN,NaN,b02765,265,unknown,unknown,b02765,5,10,4
18706698,2015-05-12 20:10:00,NaN,NaN,b02764,265,unknown,unknown,b01197,5,20,1
18709006,2015-05-12 20:36:00,NaN,NaN,b02764,265,unknown,unknown,b00014,5,20,1
18709671,2015-02-20 08:39:03,NaN,NaN,b02765,265,unknown,unknown,b02765,2,8,4


In [7]:
df = df.dropna(subset=['lon', 'lat'])

In [8]:
missing_values = df.isnull().sum()/df.shape[0]*100
print(f"df : \n{missing_values}")

df : 
pickup_date              0.00000
lat                      0.00000
lon                      0.00000
dispatching_base_num     0.00000
locationid               0.00000
borough                  0.00000
zone                     0.00000
affiliated_base_num     23.75209
month                    0.00000
hour                     0.00000
dayofweek                0.00000
dtype: float64


# Create sample

In [9]:
# begin = pd.Timestamp('2014-04-01 00:00:00')
# end = begin + pd.Timedelta(hours=1)

# s1 = pickup_14[(pickup_14['date/time'] >= begin) & (pickup_14['date/time'] < end)]
# s1

In [10]:
# s1.to_csv("c/s1.csv", index=False)

In [11]:
begin = pd.Timestamp('2015-05-17 10:00:00')
end = begin + pd.Timedelta(hours=1)

t1 = df[(df['pickup_date'] >= begin) & (df['pickup_date'] < end)]
t1.head(1)

,pickup_date,lat,lon,dispatching_base_num,locationid,borough,zone,affiliated_base_num,month,hour,dayofweek
4443324,2015-05-17 10:00:00,40.756729,-73.965146,b02617,229,manhattan,sutton place/turtle bay north,b02617,5,10,6


In [12]:
print(t1["borough"].nunique(), t1["zone"].nunique())

5 204


In [13]:
t1.to_csv("c/t1.csv", index=False)

# Remove useless variables

In [14]:
# On jette les colonnes inutiles
useless_cols = ['locationid', 'pickup_date']

print("Les colonnes suivantes vont être jetées : ", useless_cols)
t1 = t1.drop(useless_cols, axis=1)
print("...Terminé.")
print(df.head())
t1.head(1)

Les colonnes suivantes vont être jetées :  ['locationid', 'pickup_date']
...Terminé.
          pickup_date      lat      lon dispatching_base_num  locationid  \
0 2014-04-01 00:11:00  40.7690 -73.9549               b02512         140   
1 2014-04-01 00:21:00  40.7316 -73.9873               b02512          79   
2 2014-04-01 00:28:00  40.7588 -73.9776               b02512         161   
3 2014-04-01 00:33:00  40.7594 -73.9722               b02512         162   
4 2014-04-01 00:39:00  40.7223 -73.9887               b02512         148   

     borough             zone affiliated_base_num  month  hour  dayofweek  
0  manhattan  lenox hill east                 NaN      4     0          1  
1  manhattan     east village                 NaN      4     0          1  
2  manhattan   midtown center                 NaN      4     0          1  
3  manhattan     midtown east                 NaN      4     0          1  
4  manhattan  lower east side                 NaN      4     0          1  


,lat,lon,dispatching_base_num,borough,zone,affiliated_base_num,month,hour,dayofweek
4443324,40.756729,-73.965146,b02617,manhattan,sutton place/turtle bay north,b02617,5,10,6


# Normalize dataset

Before applying K-Means, let's normalize our dataset to avoid inflating our dataset since K-Means is using Euclidian Distance.

In [15]:
# Création du pipeline pour les variables quantitatives
numeric_features = ['lon', 'lat', 'month', 'hour', 'dayofweek'] # Positions des colonnes quantitatives dans X
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler()) # pour normaliser les variables
])

# Création du pipeline pour les variables catégorielles
categorical_features = ['dispatching_base_num', 'affiliated_base_num',	'borough',	'zone'] # Positions des colonnes catégorielles dans X
categorical_transformer = Pipeline(
    steps=[
    ('encoder', OneHotEncoder()) # on encode les catégories sous forme de colonnes comportant des 0 et des 1
    ])

# On combine les pipelines dans un ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessings sur le dataset
print("Preprocessing sur le train set...")
print(t1.head())
X = preprocessor.fit_transform(t1) # fit_transform !!
print('...Terminé.')
print(X[0:5, :])
print()

Preprocessing sur le train set...
               lat        lon dispatching_base_num    borough  \
4443324  40.756729 -73.965146               b02617  manhattan   
4443325  40.766948 -73.959635               b02617  manhattan   
4443326  40.745379 -73.948892               b02617     queens   
4443327  40.718804 -73.957418               b02617   brooklyn   
4443328  40.766948 -73.959635               b02617  manhattan   

                                   zone affiliated_base_num  month  hour  \
4443324   sutton place/turtle bay north              b02617      5    10   
4443325                 lenox hill west              b02617      5    10   
4443326  long island city/hunters point              b02765      5    10   
4443327       williamsburg (north side)              b02774      5    10   
4443328                 lenox hill west              b02617      5    10   

         dayofweek  
4443324          6  
4443325          6  
4443326          6  
4443327          6  
4443328      

In [16]:
# # Import K-Means
# from sklearn.cluster import KMeans

# pts = t1.groupby(["lat", "lon"]).size().reset_index(name="n")
# kmeans = KMeans(n_clusters=4, random_state=42).fit(pts[["lat", "lon"]], sample_weight=pts["n"])
# pts["cluster"] = kmeans.labels_

In [17]:
from sklearn.cluster import KMeans

# Instanciate KMeans with k=4 and initialisation with k-means++
# You should always use k-means++ as it alleviate the problem of local minimum convergence
kmeans = KMeans(n_clusters=4, random_state=42)

# Fit kmeans to our dataset
kmeans.fit(X)

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",4
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",'auto'
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",42
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [18]:
data = t1.assign(cluster=kmeans.labels_.astype(str)) # transform les cluster en labels
centers = data.groupby("cluster")[["lat", "lon"]].median().reset_index()
fig = px.scatter_map(data, lat="lat", lon="lon", color="cluster", zoom=9, map_style="carto-positron", category_orders={"cluster": ["0", "1", "2", "3"]})
fig.add_scattermap(lat=centers["lat"], lon=centers["lon"], mode="markers", marker=dict(size=12, color="black"), name="centres")
fig.show()